# Data Validation

Validate all input data, check schema compliance, referential integrity, null analysis, and distribution sanity checks.

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = Path("../../data/processed")
INTERMEDIATE_DIR = Path("../../data/intermediate")

from buildanalysis.loading import BuildDataset, _SCHEMA_MAP
ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)

CORE_FILES = ["file_metrics", "target_metrics", "edge_list", "contributor_target_commits"]
STAGE0_FILES = ["git_commit_log", "header_edges", "header_metrics", "build_schedule"]
ALL_FILES = CORE_FILES + STAGE0_FILES

## 1. Load & Inventory

In [ ]:
loaded = {}
file_status = {}

for name in ALL_FILES:
    exists = ds.has_file(name)
    status = {"exists": exists, "rows": None, "cols": None}
    if exists:
        try:
            df = pd.read_parquet(DATA_DIR / f"{name}.parquet")
            loaded[name] = df
            status["rows"] = len(df)
            status["cols"] = len(df.columns)
        except Exception as e:
            status["error"] = str(e)
    file_status[name] = status

core_present = sum(1 for n in CORE_FILES if file_status[n]["exists"])
stage0_present = sum(1 for n in STAGE0_FILES if file_status[n]["exists"])

print("FILE INVENTORY")
print("=" * 70)
print(f"{'File':<40} {'Exists':<8} {'Rows':<12} {'Cols':<6}")
print("-" * 70)
for name in ALL_FILES:
    s = file_status[name]
    exists_str = "Y" if s["exists"] else "N"
    rows_str = f"{s['rows']:,}" if s['rows'] is not None else "-"
    cols_str = str(s['cols']) if s['cols'] is not None else "-"
    print(f"{name:<40} {exists_str:<8} {rows_str:<12} {cols_str:<6}")
print("-" * 70)
print(f"Core files: {core_present}/{len(CORE_FILES)} present")
print(f"Stage 0 files: {stage0_present}/{len(STAGE0_FILES)} present")

## 2. Schema Validation

In [ ]:
import pandera.pandas as pa

schema_results = {}
print("\nSCHEMA VALIDATION")
print("=" * 70)

for name in ALL_FILES:
    if name not in loaded:
        schema_results[name] = {"status": "SKIPPED", "reason": "file not loaded"}
        print(f"  {name:<40} SKIPPED (file not loaded)")
        continue

    if name not in _SCHEMA_MAP:
        schema_results[name] = {"status": "SKIPPED", "reason": "no schema defined"}
        print(f"  {name:<40} SKIPPED (no schema)")
        continue

    schema_cls = _SCHEMA_MAP[name]
    try:
        schema_cls.validate(loaded[name])
        schema_results[name] = {"status": "PASSED"}
        print(f"  {name:<40} PASSED")
    except pa.errors.SchemaError as e:
        schema_results[name] = {"status": "FAILED", "error": str(e)}
        print(f"  {name:<40} FAILED")
        for line in str(e).splitlines()[:3]:
            print(f"    {line}")

print("-" * 70)
passed = sum(1 for v in schema_results.values() if v["status"] == "PASSED")
failed = sum(1 for v in schema_results.values() if v["status"] == "FAILED")
skipped = sum(1 for v in schema_results.values() if v["status"] == "SKIPPED")
print(f"Passed: {passed}  Failed: {failed}  Skipped: {skipped}")

## 3. Referential Integrity

In [ ]:
integrity_issues = []

def check_fk(child_name, child_col, parent_name, parent_col, label=None):
    """Check that all values in child_col exist in parent_col."""
    label = label or f"{child_name}.{child_col} -> {parent_name}.{parent_col}"

    if child_name not in loaded or parent_name not in loaded:
        print(f"  {label}: SKIPPED (missing data)")
        return

    child_vals = set(loaded[child_name][child_col].dropna().unique())
    parent_vals = set(loaded[parent_name][parent_col].dropna().unique())
    missing = child_vals - parent_vals

    total = len(child_vals)
    n_ok = total - len(missing)

    if missing:
        rows_affected = loaded[child_name][child_col].isin(missing).sum()
        msg = f"{rows_affected} {child_name} rows reference unknown {parent_col} values"
        integrity_issues.append(msg)
        print(f"  {label}")
        print(f"    Checked: {total}  Passing: {n_ok}  Failing: {len(missing)}  Rows: {rows_affected}")
    else:
        print(f"  {label}")
        print(f"    Checked: {total}  PASSED")

print("\nREFERENTIAL INTEGRITY")
print("=" * 70)

check_fk("file_metrics", "cmake_target", "target_metrics", "cmake_target")
check_fk("edge_list", "source_target", "target_metrics", "cmake_target")
check_fk("edge_list", "dest_target", "target_metrics", "cmake_target")
check_fk("contributor_target_commits", "cmake_target", "target_metrics", "cmake_target")

print("-" * 70)
if integrity_issues:
    print(f"{len(integrity_issues)} integrity issue(s) found:")
    for issue in integrity_issues:
        print(f"  - {issue}")
else:
    print("All referential integrity checks passed")

## 4. Null Analysis

In [ ]:
null_flags = []

print("\nNULL ANALYSIS")
print("=" * 70)

for name in ALL_FILES:
    if name not in loaded:
        continue

    df = loaded[name]
    null_counts = df.isnull().sum()
    null_pct = 100 * null_counts / len(df)
    has_nulls = null_counts[null_counts > 0]

    print(f"\n--- {name} ({len(df):,} rows) ---")
    if has_nulls.empty:
        print("  No null values")
        continue

    report = pd.DataFrame({
        "null_count": has_nulls,
        "null_pct": null_pct[has_nulls.index].round(1),
    }).sort_values("null_pct", ascending=False)

    for col, row in report.iterrows():
        flag = " <<<" if row["null_pct"] > 10 else ""
        print(f"  {col:<45} {int(row['null_count']):>8} ({row['null_pct']:>5.1f}%){flag}")
        if row["null_pct"] > 10:
            null_flags.append(f"{name}.{col}: {row['null_pct']:.0f}% null")

print("\n" + "=" * 70)
if null_flags:
    print(f"{len(null_flags)} column(s) with >10% nulls:")
    for flag in null_flags:
        print(f"  - {flag}")
else:
    print("No columns with >10% nulls")

## 5. Distribution Sanity Checks

In [ ]:
distribution_issues = []

RANGE_CHECKS = [
    ("file_metrics", "compile_time_ms", 0, 600_000, "compile time (ms)"),
    ("file_metrics", "code_lines", 0, 100_000, "lines of code"),
    ("file_metrics", "preprocessed_bytes", 0, None, "preprocessed size"),
    ("target_metrics", "total_build_time_ms", 0, None, "total build time"),
    ("target_metrics", "betweenness_centrality", 0, 1, "betweenness centrality"),
]

print("\nDISTRIBUTION SANITY CHECKS")
print("=" * 70)

for file_name, col, lo, hi, desc in RANGE_CHECKS:
    if file_name not in loaded or col not in loaded[file_name].columns:
        print(f"  {file_name}.{col}: SKIPPED (not available)")
        continue

    series = loaded[file_name][col].dropna()
    if series.empty:
        print(f"  {file_name}.{col}: SKIPPED (all null)")
        continue

    s_min = series.min()
    s_max = series.max()
    s_mean = series.mean()

    violations = 0
    if lo is not None:
        violations += (series < lo).sum()
    if hi is not None:
        violations += (series > hi).sum()

    status = "PASSED" if violations == 0 else f"FAILED ({violations} violations)"
    if violations > 0:
        distribution_issues.append(f"{file_name}.{col}: {violations} values outside [{lo}, {hi}]")

    print(f"  {file_name}.{col} — {desc}")
    print(f"    min={s_min:.2e}  max={s_max:.2e}  mean={s_mean:.2e}  violations={violations}")

print("\n--- Zero-variance columns ---")
zero_var_found = False
for name in ALL_FILES:
    if name not in loaded:
        continue
    df = loaded[name]
    numeric_cols = df.select_dtypes(include="number").columns
    for col in numeric_cols:
        non_null = df[col].dropna()
        if len(non_null) > 1 and non_null.nunique() == 1:
            print(f"  {name}.{col} = {non_null.iloc[0]} (constant)")
            zero_var_found = True

if not zero_var_found:
    print("  No zero-variance numeric columns found")

print("\n" + "=" * 70)
if distribution_issues:
    print(f"{len(distribution_issues)} distribution issue(s):")
    for issue in distribution_issues:
        print(f"  - {issue}")
else:
    print("All distribution checks passed")

## 6. Overall Assessment

In [ ]:
print("\n" + "=" * 70)
print("DATA VALIDATION SUMMARY")
print("=" * 70)

core_ok = all(file_status[n]["exists"] for n in CORE_FILES)
core_str = f"{core_present}/{len(CORE_FILES)} present"
print(f"Core files: {core_str} {'✓' if core_ok else '✗ BLOCKING'}")

stage0_str = f"{stage0_present}/{len(STAGE0_FILES)} present"
print(f"Stage 0 files: {stage0_str}")

schema_passed = all(
    schema_results.get(n, {}).get("status") in ("PASSED", "SKIPPED")
    for n in ALL_FILES
)
schema_failed_names = [
    n for n in ALL_FILES
    if schema_results.get(n, {}).get("status") == "FAILED"
]
if schema_passed:
    print("Schema validation: All passed ✓")
else:
    print(f"Schema validation: {len(schema_failed_names)} FAILED ✗")

if integrity_issues:
    print(f"Referential integrity: {len(integrity_issues)} issue(s) ⚠")
else:
    print("Referential integrity: All passed ✓")

if null_flags:
    print(f"Null analysis: {len(null_flags)} column(s) with >10% nulls ⚠")
else:
    print("Null analysis: All passed ✓")

if distribution_issues:
    print(f"Distribution checks: {len(distribution_issues)} issue(s) ⚠")
else:
    print("Distribution checks: All passed ✓")

print()
blocking = not core_ok or not schema_passed
if blocking:
    print("ASSESSMENT: BLOCKING issues found. Fix before proceeding with analysis.")
elif integrity_issues or null_flags or distribution_issues:
    print("ASSESSMENT: Proceed with analysis. Minor issues are non-blocking.")
else:
    print("ASSESSMENT: All checks passed. Data is ready for analysis.")
print("=" * 70)